## Bangla VITS (mobassir94) — GPU test

Single-speaker Bangla VITS models (male & female, Apache 2.0) from [comprehensive-bangla-tts](https://github.com/mobassir94/comprehensive-bangla-tts), run with [coqui-tts](https://github.com/idiap/coqui-ai-TTS).

**Runtime → Change runtime type → T4 GPU**, then run all cells.

RTF = generation time / audio length. Lower is faster; below 1.0 is faster than real time.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q "coqui-tts[bn,codec]" soundfile onnx onnxscript onnxruntime

### Download models
Each zip is ~900 MB (full training checkpoint). The optimizer state is stripped after download.

In [ ]:
import os, torch

MODEL_ROOT = "/content/model"
RELEASE_URL = "https://github.com/coqui-ai/TTS/releases/download/v0.13.3_models"
VOICES = {"male": "tts_models--bn--custom--vits_male", "female": "tts_models--bn--custom--vits_female"}

os.makedirs(MODEL_ROOT, exist_ok=True)
for voice, name in VOICES.items():
    out_dir = f"{MODEL_ROOT}/vits-{voice}"
    if os.path.isfile(f"{out_dir}/model_file.pth"):
        continue
    !wget -q --show-progress -O {MODEL_ROOT}/{name}.zip {RELEASE_URL}/{name}.zip
    !unzip -q -o {MODEL_ROOT}/{name}.zip -d {MODEL_ROOT} && mv {MODEL_ROOT}/{name} {out_dir} && rm {MODEL_ROOT}/{name}.zip
    state = torch.load(f"{out_dir}/model_file.pth", map_location="cpu", weights_only=False)
    torch.save({"model": state["model"]}, f"{out_dir}/model_file.pth")
    del state

!du -sh {MODEL_ROOT}/*

### Text normalization + generation helpers (same as `BanglaVITS/main.py`)

In [ ]:
import re, time
import numpy as np
import soundfile as sf
from IPython.display import Audio, display
from bangla import convert_english_digit_to_bangla_digit
from bnnumerizer import numerize
from bnunicodenormalizer import Normalizer
from TTS.api import TTS

OUTPUT_DIR = "/content/outputs"
SENTENCE_PAUSE = 0.25
_bnorm = Normalizer()


def normalize_text(text):
    text = convert_english_digit_to_bangla_digit(text)
    text = numerize(text)
    text = text.replace("\n", " ").replace(";", ",").replace(":", ",")
    words = [_bnorm(w)["normalized"] for w in text.split()]
    return " ".join(w for w in words if w)


def split_sentences(text):
    sentences = []
    for p in re.split(r"(?<=[।!?])\s*", text):
        p = p.strip(" ,")
        if p:
            sentences.append(p if p[-1] in "।!?" else p + "।")
    return sentences


def load_model(voice, device, use_onnx=False):
    d = f"{MODEL_ROOT}/vits-{voice}"
    start = time.time()
    m = TTS(model_path=f"{d}/model_file.pth", config_path=f"{d}/config.json", progress_bar=False).to(device)
    m.use_onnx = use_onnx and device == "cpu"
    if m.use_onnx:
        if not os.path.isfile(f"{d}/model.onnx"):
            m.synthesizer.tts_model.export_onnx(output_path=f"{d}/model.onnx", verbose=False)
        m.synthesizer.tts_model.load_onnx(f"{d}/model.onnx")
    print(f"{voice} loaded on {device} ({'onnx' if m.use_onnx else 'torch'}) in {time.time() - start:.1f}s")
    return m


def synth(model, sentence):
    vits = model.synthesizer.tts_model
    if model.use_onnx:
        ids = np.array([vits.tokenizer.text_to_ids(sentence)], dtype=np.int64)
        return vits.inference_onnx(ids).squeeze()
    return model.tts(sentence, split_sentences=False)


def generate(model, text, filename, speed=1.0, play=True):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    sr = model.synthesizer.output_sample_rate
    model.synthesizer.tts_model.length_scale = 1.0 / speed
    silence = np.zeros(int(SENTENCE_PAUSE * sr), dtype=np.float32)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start = time.time()
    chunks = []
    with torch.inference_mode():
        for s in split_sentences(normalize_text(text)):
            chunks += [np.asarray(synth(model, s), dtype=np.float32), silence]
    audio = np.concatenate(chunks[:-1])
    elapsed = time.time() - start
    duration = len(audio) / sr
    path = f"{OUTPUT_DIR}/{filename}.wav"
    sf.write(path, audio, sr)
    print(f"{filename}: audio {duration:.1f}s | generated in {elapsed:.2f}s | RTF {elapsed / duration:.3f}")
    if play:
        display(Audio(path))
    return {"file": filename, "audio_s": round(duration, 2), "gen_s": round(elapsed, 3), "rtf": round(elapsed / duration, 3)}

### Test sentences

In [ ]:
TESTS = {
    "word": "ধন্যবাদ",
    "sentence1": "আমার সোনার বাংলা, আমি তোমায় ভালোবাসি।",
    "sentence2": "আজকের আবহাওয়া খুব সুন্দর, চলো আমরা বাইরে ঘুরতে যাই।",
    "numbers": "আমার বয়স 25 বছর, আর আজ ২৪ তারিখ।",
    "question": "তুমি কেমন আছো? আমি ভালো আছি!",
    "paragraph": (
        "বাংলাদেশ দক্ষিণ এশিয়ার একটি সুন্দর দেশ। এই দেশের মানুষ খুবই অতিথিপরায়ণ। "
        "এখানে ছয়টি ঋতু আছে, এবং প্রতিটি ঋতুর নিজস্ব সৌন্দর্য রয়েছে। "
        "নদীমাতৃক এই দেশে পদ্মা, মেঘনা ও যমুনা প্রধান নদী। "
        "ঢাকা বাংলাদেশের রাজধানী এবং সবচেয়ে বড় শহর।"
    ),
}

### GPU

In [ ]:
gpu_results = []
for voice in ["female", "male"]:
    model = load_model(voice, "cuda")
    synth(model, "ওয়ার্ম আপ।")  # warm-up, not timed
    for name, text in TESTS.items():
        r = generate(model, text, f"gpu_{voice}_{name}")
        gpu_results.append({"device": "gpu", "voice": voice, **r})
    del model
    torch.cuda.empty_cache()

### CPU: PyTorch vs ONNX (Colab has only ~2 vCPUs, so your server will likely be faster)

In [ ]:
cpu_results = []
for voice in ["female", "male"]:
    for use_onnx in [False, True]:
        backend = "cpu-onnx" if use_onnx else "cpu-torch"
        model = load_model(voice, "cpu", use_onnx=use_onnx)
        synth(model, "ওয়ার্ম আপ।")
        for name, text in TESTS.items():
            r = generate(model, text, f"{backend}_{voice}_{name}", play=False)
            cpu_results.append({"device": backend, "voice": voice, **r})
        del model

### Summary

In [ ]:
import pandas as pd
df = pd.DataFrame(gpu_results + cpu_results)
display(df)
display(df.groupby(["device", "voice"])[["audio_s", "gen_s"]].sum().assign(rtf=lambda d: (d.gen_s / d.audio_s).round(3)))

### Try your own text

In [ ]:
model = load_model("female", "cuda")  # or "male"
generate(model, "এখানে আপনার নিজের বাংলা লেখা দিন।", "custom", speed=1.0)